   
### Bronze to Silver -- HR Domain
**Author:** Virendra Tambavekar  
**Task:** Transform HR domain data from Bronze (ALL STRING) to Silver (typed, deduped)  
**Domain:** HR (Broker)  
**Pipeline Stage:** Bronze -> Silver  
**Source:** `charles_schwab_retailbrokerage_dev_team_lemma.bronze.hr` (50,000 rows)  
**Target:** `charles_schwab_retailbrokerage_dev_team_lemma.silver.broker` (50,000 rows)  

**Transformations:**
- Type cast: EMPLOYEE_ID, MANAGER_ID, JOB_CODE, BRANCH_ID -> INT
- Build FULL_NAME from FIRST_NAME + MIDDLE_INITIAL + LAST_NAME
- Dedup by EMPLOYEE_ID (latest `_ingest_ts` wins)
- Carry-forward `_run_id` from upstream bronze layer (lineage tracking)
- Silver audit columns: `_load_ts`, `_batch`, `_run_id`
- Mode: CREATE OR REPLACE (full rebuild)
- Operations logging: pipeline recon + audit event

In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Configuration
CATALOG = "charles_schwab_retailbrokerage_dev_team_lemma"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

SOURCE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.hr"
TARGET_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.broker"

print(f"Source: {SOURCE_TABLE}")
print(f"Target: {TARGET_TABLE}")

In [0]:
# Read from Bronze (ALL STRING columns)
bronze_df = spark.table(SOURCE_TABLE)
print(f"Bronze row count: {bronze_df.count()}")
bronze_df.printSchema()

In [0]:
# Extract carry-forwarded run_id from Bronze layer
spark.sql(f"USE CATALOG {CATALOG}")
carried_run_id = str(bronze_df.select("`_run_id`").first()[0])
print(f"Carry-forwarded run_id: {carried_run_id}")

# __ start_pipeline_run -- imported from operations
start_pipeline_run(spark=spark, run_id=carried_run_id, batch="ALL")

# __ log_domain_run_status -- imported from operations
log_domain_run_status(spark=spark, run_id=carried_run_id, batch="ALL", domain_name="HR", status="RUNNING")

# __ log_pipeline_message -- imported from operations
log_pipeline_message(spark=spark, run_id=carried_run_id, log_level="INFO", module="bronze_to_silver_hr", message="Pipeline started: Bronze to Silver transformation for HR domain")

In [0]:
# Step 1: Dedup by EMPLOYEE_ID (latest _ingest_ts wins)
dedup_window = Window.partitionBy("EMPLOYEE_ID").orderBy(F.col("_ingest_ts").desc())

deduped_df = (
    bronze_df
    .withColumn("_row_num", F.row_number().over(dedup_window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

print(f"After dedup: {deduped_df.count()} rows")

In [0]:
# Step 2: Type casting, FULL_NAME construction, and carry-forward _run_id
# Using try_cast to handle malformed values (returns NULL instead of error)
silver_df = (
    deduped_df
    .select(
        F.expr("TRY_CAST(EMPLOYEE_ID AS INT)").alias("EMPLOYEE_ID"),
        F.expr("TRY_CAST(MANAGER_ID AS INT)").alias("MANAGER_ID"),
        F.col("LAST_NAME"),
        F.col("FIRST_NAME"),
        F.col("MIDDLE_INITIAL"),
        F.expr("TRY_CAST(JOB_CODE AS INT)").alias("JOB_CODE"),
        F.expr("TRY_CAST(BRANCH_ID AS INT)").alias("BRANCH_ID"),
        F.col("OFFICE"),
        F.col("PHONE"),
        # Build FULL_NAME: FIRST_NAME + MIDDLE_INITIAL + LAST_NAME
        F.trim(
            F.concat_ws(" ",
                F.col("FIRST_NAME"),
                F.when(F.col("MIDDLE_INITIAL").isNotNull() & (F.col("MIDDLE_INITIAL") != ""), F.col("MIDDLE_INITIAL")),
                F.col("LAST_NAME")
            )
        ).alias("FULL_NAME"),
        # Carry-forward _run_id from upstream bronze layer
        F.col("_run_id"),
        F.col("_batch_id").alias("_batch"),
        F.current_timestamp().alias("_load_ts")
    )
)

print("Silver DataFrame schema:")
silver_df.printSchema()
print(f"Silver row count: {silver_df.count()}")

In [0]:
# Step 3: Write to Silver as CREATE OR REPLACE (full rebuild)
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TARGET_TABLE)

print(f"Silver table written successfully: {TARGET_TABLE}")

In [0]:
# Step 4: Validation -- verify row counts and sample data
target_count = spark.table(TARGET_TABLE).count()
source_count = bronze_df.count()

print(f"Source (Bronze): {source_count} rows")
print(f"Target (Silver): {target_count} rows")
print(f"Status: {'MATCH' if source_count == target_count else 'MISMATCH'}")
print("\n--- Sample Data ---")
display(spark.table(TARGET_TABLE).limit(10))

In [0]:
# Step 5: Operations Logging

# __ log_pipeline_recon -- imported from operations
log_pipeline_recon(
    spark=spark,
    run_id=carried_run_id,
    batch_id="ALL",
    domain="HR",
    table_name="broker",
    source_layer="bronze",
    target_layer="silver",
    source_count=source_count,
    target_count=target_count
)

# __ log_audit_event -- imported from operations
log_audit_event(
    spark=spark,
    run_id=carried_run_id,
    batch="ALL",
    layer="silver",
    table_name="broker",
    operation="OVERWRITE",
    rows_affected=target_count
)

# __ log_pipeline_message -- imported from operations
log_pipeline_message(spark=spark, run_id=carried_run_id, log_level="INFO", module="bronze_to_silver_hr", message=f"Pipeline completed: {target_count} rows written to silver.broker")

# __ log_domain_run_status -- imported from operations
log_domain_run_status(spark=spark, run_id=carried_run_id, batch="ALL", domain_name="HR", status="COMPLETED")

# __ end_pipeline_run -- imported from operations
end_pipeline_run(spark=spark, run_id=carried_run_id, status="SUCCESS")

print(f"Operations logging complete for run_id: {carried_run_id}")

In [0]:
def log_dq_result(spark: SparkSession, run_id: str, table_name: str, rule_name: str, failed_rows: int, total_rows: int):
    """
    Logs the outcome of a Data Quality (DQ) check.
    """
    status = 'PASS' if failed_rows == 0 else 'FAIL'
    
    spark.sql(f"""
        INSERT INTO operations.dq_results 
        (run_id, table_name, rule_name, failed_rows, total_rows, dq_status)
        VALUES ('{run_id}', '{table_name}', '{rule_name}', {failed_rows}, {total_rows}, '{status}')
    """)